# YOLO Ailesi Karşılaştırması (v8, v11, v12) — Cilt Kanseri Sınıflandırması

Proje önerinizin **ana özgün katkısı:** YOLOv8-v12 modellerinin cilt lezyonu sınıflandırmasında sistematik karşılaştırılması.

**Not:** YOLOv9-cls ve YOLOv10-cls için Ultralytics resmi sınıflandırma varyantı bulunmuyor (sadece detection modeli var). Bu yüzden YOLOv8m-cls, YOLO11m-cls ve YOLO12m-cls karşılaştırılacak. TÜBİTAK raporunda bu sınırlılığı belirtirsiniz:
> *"YOLOv9 ve YOLOv10 ailesinde Ultralytics resmi kütüphanesinde sınıflandırma varyantı bulunmadığı için, karşılaştırma YOLOv8m-cls, YOLO11m-cls ve YOLO12m-cls modelleri üzerinden yapıldı."*

**Akış:**
1. Ortam kurulumu + Drive mount
2. Veri açma + Ultralytics classification formatına dönüşüm (class-per-folder)
3. Minority oversampling (symlink ile, disk artmıyor)
4. Her model için ardışık eğitim (~1 saat/model, toplam ~3-4 saat)
5. Her modelde manuel inference + sklearn metrikleri
6. Karşılaştırma tablosu
7. Pairwise McNemar testi (istatistiksel anlamlılık)
8. En iyi modelin ONNX export'u

**Beklenen sonuçlar:**
- YOLOv8m-cls: F1 ≈ 0.78-0.82
- YOLO11m-cls: F1 ≈ 0.80-0.84 (C3k2 + C2PSA dikkat)
- YOLO12m-cls: F1 ≈ 0.82-0.86 (Area Attention, R-ELAN)
- Hipotez 1 testi: YOLOv12 melanom duyarlılığında diğerlerinden üstün mü?

## 0. Ortam kurulumu

In [ ]:
!nvidia-smi

In [ ]:
# Ultralytics 8.3+ YOLO v8/11/12 classification destekler
!pip install -q ultralytics==8.3.40 scikit-learn matplotlib seaborn pandas
# Check ultralytics version
!python -c "import ultralytics; print('Ultralytics:', ultralytics.__version__)"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch

DRIVE_DATA  = '/content/drive/MyDrive/Colab Notebooks/CiltKanseri/CiltKanseriProjesi'
DRIVE_MINE  = '/content/drive/MyDrive/Colab Notebooks/CiltKanseri/CiltKanseriProje'
LOCAL_ROOT  = '/content/dataset'
IMG_ROOT    = f'{LOCAL_ROOT}/dataset_yolo/images'
CLS_ROOT    = '/content/cls_data'   # Ultralytics için class-per-folder yapı

YOLO_RUNS = f'{DRIVE_MINE}/yolo_runs'
os.makedirs(YOLO_RUNS, exist_ok=True)

CFG = {
    'class_names': ['nv', 'mel', 'bcc', 'bkl', 'akiec', 'vasc', 'df'],
    'img_size': 384,
    'batch_size': 128,             # A100'de cls modu için rahat
    'epochs': 50,
    'patience': 15,
    'lr0': 1e-3,                   # Ultralytics cls default
    'lrf': 0.01,
    'weight_decay': 1e-4,
    'label_smoothing': 0.1,
    'mixup': 0.1,
    'oversample_min': 3000,        # minority sınıfları 3000'e kadar oversample
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device} | GPU: {torch.cuda.get_device_name(0) if device == "cuda" else "-"}')
print(f'Runs: {YOLO_RUNS}')

## 1. Veri açma (zip açıksa atlar)

In [ ]:
if not os.path.exists(f'{IMG_ROOT}/train'):
    os.makedirs(LOCAL_ROOT, exist_ok=True)
    t0 = time.time()
    print('Zip açılıyor... (~15-20 dk)')
    !unzip -q "$DRIVE_DATA/dataset_yolo.zip" -d /content/dataset
    print(f'Tamamlandı: {(time.time()-t0)/60:.1f} dakika')
else:
    print('Veri zaten açık.')

for split in ['train', 'val', 'test']:
    n = len(os.listdir(f'{IMG_ROOT}/{split}'))
    print(f'{split:5s}: {n:6d} dosya')

## 2. Ultralytics class-per-folder formatına dönüşüm (symlink, hızlı)

Ultralytics classification `data/train/{class}/img.jpg` bekliyor. Sembolik link ile 9GB'ı kopyalamadan yeniden düzenliyoruz. Minority sınıfları (vasc=194, df=180) için **oversampling** yapıyoruz — her sınıfın en az 3000 örneği olsun diye linkleri tekrarlıyoruz.

In [ ]:
def restructure(src_root, dst_root, oversample_target=None):
    """
    Kaynak yapı: {src_root}/{split}/{class}_{image}.jpg
    Hedef yapı:  {dst_root}/{split}/{class}/{class}_{image}.jpg  (symlink)
    oversample_target: sadece train'e uygulanır; her sınıfı en az bu sayıya çıkarır (linkleri tekrarlar)
    """
    if os.path.exists(dst_root):
        shutil.rmtree(dst_root)
    os.makedirs(dst_root)

    counts_report = {}
    for split in ['train', 'val', 'test']:
        src = Path(src_root) / split
        # sınıf → dosya listesi
        class_files = {}
        for f in src.iterdir():
            if not f.is_file():
                continue
            cls = f.stem.split('_')[0]  # nv_ISIC_XXX → nv
            class_files.setdefault(cls, []).append(f)

        counts_report[split] = {c: len(fs) for c, fs in class_files.items()}

        # Symlink'leri oluştur
        for cls, files in class_files.items():
            dst_dir = Path(dst_root) / split / cls
            dst_dir.mkdir(parents=True, exist_ok=True)

            # Orijinal linkler
            for f in files:
                target = dst_dir / f.name
                if not target.exists():
                    os.symlink(f.resolve(), target)

            # Train'de oversampling — aynı dosyaya farklı isimli linkler
            if split == 'train' and oversample_target is not None:
                n_cur = len(files)
                n_extra = max(0, oversample_target - n_cur)
                for i in range(n_extra):
                    src_file = files[i % n_cur]
                    link_name = f'os{i:05d}_{src_file.name}'
                    link_path = dst_dir / link_name
                    if not link_path.exists():
                        os.symlink(src_file.resolve(), link_path)

    return counts_report

print('Dönüşüm başlıyor...')
t0 = time.time()
orig_counts = restructure(IMG_ROOT, CLS_ROOT, oversample_target=CFG['oversample_min'])
print(f'Tamamlandı: {time.time()-t0:.0f} saniye')

# Raporlama
print('\n=== ORİJİNAL DAĞILIM ===')
for split, counts in orig_counts.items():
    print(f'\n{split}:')
    for c in CFG['class_names']:
        print(f'  {c:8s} {counts.get(c, 0):>6d}')

print('\n=== OVERSAMPLING SONRASI (TRAIN) ===')
for c in CFG['class_names']:
    dst_dir = Path(CLS_ROOT) / 'train' / c
    n_final = len(list(dst_dir.iterdir())) if dst_dir.exists() else 0
    print(f'  {c:8s} {orig_counts["train"].get(c, 0):>6d} → {n_final:>6d}')

## 3. Her YOLO modeli için eğitim + değerlendirme fonksiyonu

Ultralytics classification API'sı yerleşik `top1_acc` veriyor ama bizim ihtiyacımız olan macro F1, mel_recall, Kappa'yı manuel sklearn ile hesaplıyoruz.

In [ ]:
from ultralytics import YOLO
from sklearn.metrics import (
    classification_report, confusion_matrix, cohen_kappa_score,
    f1_score, accuracy_score, recall_score, balanced_accuracy_score
)
import cv2
from torch.utils.data import Dataset, DataLoader

class TestDataset(Dataset):
    """Test/val setinde manuel inference için."""
    def __init__(self, root, class_names, img_size=384):
        self.items = []
        self.class_names = class_names
        self.c2i = {c: i for i, c in enumerate(class_names)}
        for cls_dir in Path(root).iterdir():
            if not cls_dir.is_dir():
                continue
            cls = cls_dir.name
            if cls not in self.c2i:
                continue
            for f in cls_dir.iterdir():
                self.items.append((str(f), self.c2i[cls]))
        self.img_size = img_size

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        p, y = self.items[i]
        img = cv2.imread(p)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return img, y, p

def evaluate_yolo_cls(model_weights_path, test_root, class_names, img_size=384):
    """Eğitilmiş YOLO-cls modelini test setinde manuel değerlendir."""
    model = YOLO(model_weights_path)
    ds = TestDataset(test_root, class_names, img_size=img_size)
    all_probs, all_true, all_paths = [], [], []

    for i in range(0, len(ds), 32):  # batch 32
        batch_paths = [ds.items[j][0] for j in range(i, min(i+32, len(ds)))]
        batch_y = [ds.items[j][1] for j in range(i, min(i+32, len(ds)))]
        results = model.predict(batch_paths, imgsz=img_size, verbose=False, device=0)
        for r, y, p in zip(results, batch_y, batch_paths):
            probs = r.probs.data.cpu().numpy()
            # Ultralytics kendi class order'ını kullanıyor — eşleştir
            ultra_names = r.names  # {idx: name}
            # Bizim order'a uygun permütasyon
            reorder = np.array([list(ultra_names.values()).index(c) for c in class_names])
            probs = probs[reorder]
            all_probs.append(probs)
            all_true.append(y)
            all_paths.append(p)

    probs = np.stack(all_probs)
    y_true = np.array(all_true)
    y_pred = probs.argmax(1)

    metrics = {
        'accuracy':     accuracy_score(y_true, y_pred),
        'bal_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_f1':     f1_score(y_true, y_pred, average='macro'),
        'weighted_f1':  f1_score(y_true, y_pred, average='weighted'),
        'mel_recall':   recall_score(y_true == class_names.index('mel'),
                                      y_pred == class_names.index('mel')),
        'mel_precision':(y_pred == class_names.index('mel')).sum() and
                        ((y_pred == class_names.index('mel')) & (y_true == class_names.index('mel'))).sum() /
                        max(1, (y_pred == class_names.index('mel')).sum()),
        'kappa':        cohen_kappa_score(y_true, y_pred),
    }
    return metrics, probs, y_true, y_pred

print('Evaluation fonksiyonları hazır.')

## 4. YOLOv8m-cls eğit ve değerlendir

⏱ A100'de ~1 saat

In [ ]:
print('=' * 60)
print('YOLOv8m-cls EĞİTİM BAŞLIYOR')
print('=' * 60)

t0 = time.time()
model_v8 = YOLO('yolov8m-cls.pt')
_ = model_v8.train(
    data=CLS_ROOT,
    epochs=CFG['epochs'],
    imgsz=CFG['img_size'],
    batch=CFG['batch_size'],
    patience=CFG['patience'],
    device=0,
    amp=True,
    optimizer='AdamW',
    lr0=CFG['lr0'],
    lrf=CFG['lrf'],
    cos_lr=True,
    weight_decay=CFG['weight_decay'],
    label_smoothing=CFG['label_smoothing'],
    mixup=CFG['mixup'],
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=30, flipud=0.5, fliplr=0.5,
    project=YOLO_RUNS,
    name='yolov8m_cls',
    exist_ok=True,
    save=True,
)
train_time_v8 = time.time() - t0

# Test seti üzerinde manuel değerlendirme
best_v8 = f'{YOLO_RUNS}/yolov8m_cls/weights/best.pt'
metrics_v8, probs_v8, y_true, y_pred_v8 = evaluate_yolo_cls(
    best_v8, f'{CLS_ROOT}/test', CFG['class_names'], img_size=CFG['img_size']
)

print(f'\n=== YOLOv8m-cls TEST SONUÇLARI (eğitim {train_time_v8/60:.1f} dk) ===')
for k, v in metrics_v8.items():
    print(f'  {k:15s}: {v:.4f}')

## 5. YOLO11m-cls eğit ve değerlendir

In [ ]:
print('=' * 60)
print('YOLO11m-cls EĞİTİM BAŞLIYOR')
print('=' * 60)

t0 = time.time()
model_v11 = YOLO('yolo11m-cls.pt')
_ = model_v11.train(
    data=CLS_ROOT,
    epochs=CFG['epochs'],
    imgsz=CFG['img_size'],
    batch=CFG['batch_size'],
    patience=CFG['patience'],
    device=0,
    amp=True,
    optimizer='AdamW',
    lr0=CFG['lr0'],
    lrf=CFG['lrf'],
    cos_lr=True,
    weight_decay=CFG['weight_decay'],
    label_smoothing=CFG['label_smoothing'],
    mixup=CFG['mixup'],
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=30, flipud=0.5, fliplr=0.5,
    project=YOLO_RUNS,
    name='yolo11m_cls',
    exist_ok=True,
    save=True,
)
train_time_v11 = time.time() - t0

best_v11 = f'{YOLO_RUNS}/yolo11m_cls/weights/best.pt'
metrics_v11, probs_v11, _, y_pred_v11 = evaluate_yolo_cls(
    best_v11, f'{CLS_ROOT}/test', CFG['class_names'], img_size=CFG['img_size']
)

print(f'\n=== YOLO11m-cls TEST SONUÇLARI (eğitim {train_time_v11/60:.1f} dk) ===')
for k, v in metrics_v11.items():
    print(f'  {k:15s}: {v:.4f}')

## 6. YOLO12m-cls eğit ve değerlendir (varsa)

YOLOv12 Ultralytics'e Şubat 2025'te eklendi. Eğer checkpoint indirilemezse bu hücre hata verir — o durumda atlayın.

In [ ]:
print('=' * 60)
print('YOLO12m-cls EĞİTİM BAŞLIYOR')
print('=' * 60)

metrics_v12 = None
probs_v12 = None
y_pred_v12 = None

try:
    t0 = time.time()
    model_v12 = YOLO('yolo12m-cls.pt')
    _ = model_v12.train(
        data=CLS_ROOT,
        epochs=CFG['epochs'],
        imgsz=CFG['img_size'],
        batch=CFG['batch_size'],
        patience=CFG['patience'],
        device=0,
        amp=True,
        optimizer='AdamW',
        lr0=CFG['lr0'],
        lrf=CFG['lrf'],
        cos_lr=True,
        weight_decay=CFG['weight_decay'],
        label_smoothing=CFG['label_smoothing'],
        mixup=CFG['mixup'],
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        degrees=30, flipud=0.5, fliplr=0.5,
        project=YOLO_RUNS,
        name='yolo12m_cls',
        exist_ok=True,
        save=True,
    )
    train_time_v12 = time.time() - t0

    best_v12 = f'{YOLO_RUNS}/yolo12m_cls/weights/best.pt'
    metrics_v12, probs_v12, _, y_pred_v12 = evaluate_yolo_cls(
        best_v12, f'{CLS_ROOT}/test', CFG['class_names'], img_size=CFG['img_size']
    )
    print(f'\n=== YOLO12m-cls TEST SONUÇLARI (eğitim {train_time_v12/60:.1f} dk) ===')
    for k, v in metrics_v12.items():
        print(f'  {k:15s}: {v:.4f}')
except Exception as e:
    print(f'⚠ YOLO12m-cls çalıştırılamadı: {e}')
    print('  Proje raporunda "YOLOv12 Ultralytics 8.3.x sürümünde cls varyantı henüz kararlı değildi" olarak belirtin.')
    train_time_v12 = 0

## 7. Karşılaştırma Tablosu

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

models_results = {
    'YOLOv8m-cls':  (metrics_v8,  train_time_v8),
    'YOLO11m-cls':  (metrics_v11, train_time_v11),
}
if metrics_v12 is not None:
    models_results['YOLO12m-cls'] = (metrics_v12, train_time_v12)

# DataFrame'e çevir
rows = []
for name, (m, t) in models_results.items():
    rows.append({
        'Model':       name,
        'Accuracy':    m['accuracy'],
        'Bal.Acc':     m['bal_accuracy'],
        'Macro F1':    m['macro_f1'],
        'Weighted F1': m['weighted_f1'],
        'Mel Recall':  m['mel_recall'],
        'Mel Precision': m['mel_precision'],
        'Kappa':       m['kappa'],
        'Train (dk)':  f'{t/60:.1f}',
    })
df_compare = pd.DataFrame(rows)
print('\n=== MODEL KARŞILAŞTIRMASI ===')
print(df_compare.to_string(index=False))
df_compare.to_csv(f'{YOLO_RUNS}/comparison.csv', index=False)

# Görsel: bar chart
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics_to_plot = ['macro_f1', 'mel_recall', 'kappa']
titles = ['Macro F1', 'Melanoma Recall', "Cohen's Kappa"]
for ax, metric, title in zip(axes, metrics_to_plot, titles):
    vals = [models_results[n][0][metric] for n in models_results]
    names = list(models_results.keys())
    colors = ['#3498db', '#e67e22', '#27ae60'][:len(names)]
    bars = ax.bar(names, vals, color=colors)
    ax.axhline(y=0.85, color='red', linestyle='--', alpha=0.5, label='TÜBİTAK hedef')
    ax.set_title(title); ax.set_ylim(0, 1); ax.legend()
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.02, f'{v:.3f}', ha='center')
plt.tight_layout()
plt.savefig(f'{YOLO_RUNS}/comparison.png', dpi=140)
plt.show()

## 8. McNemar Testi — İstatistiksel Anlamlılık

İki modelin test setindeki tahminleri arasında **istatistiksel olarak anlamlı** bir fark var mı? TÜBİTAK raporunda "En iyi model diğerlerinden istatistiksel olarak üstündür (p<0.05)" diyebilmek için kritik.

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

preds = {'YOLOv8m-cls': y_pred_v8, 'YOLO11m-cls': y_pred_v11}
if y_pred_v12 is not None:
    preds['YOLO12m-cls'] = y_pred_v12

print('\n=== PAIRWISE McNEMAR TEST (test seti) ===')
print(f'{"Karşılaştırma":<28} {"χ²":>8} {"p-value":>10} {"Anlamlı?":>10}')
print('-' * 60)

model_pairs = [(a, b) for i, a in enumerate(preds) for b in list(preds)[i+1:]]
for a, b in model_pairs:
    correct_a = (preds[a] == y_true)
    correct_b = (preds[b] == y_true)
    # 2x2 tablo: (a doğru?, b doğru?)
    n00 = ((~correct_a) & (~correct_b)).sum()
    n01 = ((~correct_a) &  correct_b ).sum()
    n10 = ( correct_a  & (~correct_b)).sum()
    n11 = ( correct_a  &  correct_b ).sum()
    table = [[n11, n10], [n01, n00]]
    result = mcnemar(table, exact=False, correction=True)
    sig = 'EVET (p<0.05)' if result.pvalue < 0.05 else 'hayır'
    print(f'{a+" vs "+b:<28} {result.statistic:>8.2f} {result.pvalue:>10.4f} {sig:>10}')

## 9. En iyi modelin detaylı analizi

In [ ]:
# Macro F1'e göre en iyi modeli seç
best_name = max(models_results, key=lambda k: models_results[k][0]['macro_f1'])
best_metrics = models_results[best_name][0]
print(f'\n🏆 EN İYİ MODEL: {best_name} (Macro F1 = {best_metrics["macro_f1"]:.4f})')

best_probs = {'YOLOv8m-cls': probs_v8, 'YOLO11m-cls': probs_v11}
if probs_v12 is not None:
    best_probs['YOLO12m-cls'] = probs_v12
y_pred_best = best_probs[best_name].argmax(1)

# Classification report
print('\n' + classification_report(y_true, y_pred_best, target_names=CFG['class_names'], digits=4))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred_best, normalize='true')
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CFG['class_names'], yticklabels=CFG['class_names'])
plt.ylabel('Gerçek'); plt.xlabel('Tahmin')
plt.title(f'{best_name} Test CM — F1={best_metrics["macro_f1"]:.3f}')
plt.tight_layout()
plt.savefig(f'{YOLO_RUNS}/cm_best.png', dpi=140)
plt.show()

# TÜBİTAK kriterlerine karşılaştırma
print('\n=== TÜBİTAK HEDEF KARŞILAŞTIRMASI ===')
targets = [
    ('Accuracy',     'accuracy',     0.90),
    ('Macro F1',     'macro_f1',     0.85),
    ('Mel Recall',   'mel_recall',   0.85),
    ('Cohen Kappa',  'kappa',        0.85),
]
for name, key, target in targets:
    val = best_metrics[key]
    status = '✅' if val >= target else f'❌ (-{(target-val):.3f})'
    print(f'  {name:12s} {val:.4f} / {target:.2f}  {status}')

## 10. En iyi modeli ONNX olarak export et (mobil için)

In [ ]:
best_paths = {
    'YOLOv8m-cls': f'{YOLO_RUNS}/yolov8m_cls/weights/best.pt',
    'YOLO11m-cls': f'{YOLO_RUNS}/yolo11m_cls/weights/best.pt',
    'YOLO12m-cls': f'{YOLO_RUNS}/yolo12m_cls/weights/best.pt',
}
best_pt = best_paths[best_name]
print(f'Export ediliyor: {best_pt}')

model = YOLO(best_pt)
# ONNX (FP32)
onnx_path = model.export(format='onnx', imgsz=CFG['img_size'], opset=17, simplify=True)
print(f'ONNX: {onnx_path} ({os.path.getsize(onnx_path)/1e6:.1f} MB)')

# INT8 quantize (mobil için kritik)
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType
int8_path = str(onnx_path).replace('.onnx', '_int8.onnx')
quantize_dynamic(str(onnx_path), int8_path, weight_type=QuantType.QInt8)
print(f'INT8 ONNX: {int8_path} ({os.path.getsize(int8_path)/1e6:.1f} MB) — hedef ≤ 25 MB')

## 11. TÜBİTAK Raporu için Özet

**Proje önerisi Hipotez 1 testi:**

Hipotez 1 iddiası: *"YOLOv12 modelinin, Area Attention ve R-ELAN sayesinde diğerlerine kıyasla melanom tespitinde daha yüksek duyarlılık (%85+) göstermesi beklenmektedir."*

Karşılaştırma tablosundan melanoma recall sütunu + McNemar p-değerleri ile bu hipotez test edilir.

**Rapor şablonu:**
```
Çalışmamızda YOLOv8m-cls, YOLO11m-cls ve YOLO12m-cls modelleri aynı veri seti (25,915 
ISIC 2019+2020 görüntüsü), aynı hyperparameter (AdamW, lr=1e-3, 50 epoch, cosine LR, 
MixUp α=0.1) ve aynı augmentation stratejisi ile eğitildi. Sınıf dengesizliği 
symlink-tabanlı oversampling ile (minority sınıflarını 3000'e yükselterek) dengelendi.

Sonuçlar (Tablo X):
- YOLOv8m-cls: Acc=X, F1=X, Mel-Recall=X, Kappa=X
- YOLO11m-cls: Acc=X, F1=X, Mel-Recall=X, Kappa=X  
- YOLO12m-cls: Acc=X, F1=X, Mel-Recall=X, Kappa=X

Pairwise McNemar testleri (Tablo Y): ...

Hipotez 1 [kabul/ret]: YOLOv12 melanom duyarlılığında diğerlerinden 
[istatistiksel olarak anlamlı / anlamlı olmayan] şekilde [üstün/benzer] (...)
```

**Sonraki adım:** En iyi YOLO modeli + FT-Transformer ile **multimodal füzyon** notebook'u — hasta metadata (yaş, cinsiyet, lokalizasyon) ile son %3-5 artışı alarak TÜBİTAK hedeflerini karşılamak.